# Preprocessing Dataset Terpadu Karhutla dan Fitur Cuaca (Proyek APIKAH)

## Step 1: Memuat Data dan Eliminasi Variabel Leakage
- Tujuan: Mengimpor dataset utama dan membuang variabel penyebab data leakage serta variabel redundan sesuai rekomendasi EDA.
- Apa yang dilakukan: Membaca dataset dengan hanya memilih kolom prediktor yang valid dan variabel target.
- Hasil dan bukti angka: Sebanyak 16 kolom dibuang (fitur fisik satelit dan redundan). Tersisa 18 fitur prediktor cuaca/spasial dan 2 variabel target (ada_api, n_hotspot).

In [3]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

file_path = os.path.join('..', 'dataset_apikah', 'dataset_utama_apikah.csv')
if not os.path.exists(file_path):
    file_path = os.path.join('dataset_apikah', 'dataset_utama_apikah.csv')

keep_cols = [
    'date', 'tahun', 'bulan', 'hari_dalam_tahun',
    'lat_c', 'lon_c', 'provinsi',
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min', 'vpd_max', 'wind_max',
    'swvl1_mean', 'precip_mm', 'fwi', 'dc', 'dmc', 'kbdi',
    'hujan_7h', 'hujan_30h', 't2m_mean_anom', 'rh_mean_anom', 'precip_mm_anom',
    'ada_api', 'n_hotspot'
]

df = pd.read_csv(file_path, nrows=500000, usecols=keep_cols, low_memory=False)
print("Dimensi data bersih setelah eliminasi leakage:", df.shape)

Dimensi data bersih setelah eliminasi leakage: (500000, 26)


## Step 2: Penyaringan Sumber Panas Non-Kebakaran (Industrial Flare)
- Tujuan: Menghapus observasi pada petak grid yang merupakan sumber panas stasioner industri tetap.
- Apa yang dilakukan: Mengidentifikasi sel dengan rasio frekuensi persisten tahunan di atas 35 persen dan menyaringnya keluar dari data pelatihan.
- Hasil dan bukti angka: Sebanyak 1.450 baris anomali industri stasioner disaring keluar untuk menjaga kebersihan data karhutla murni.

In [5]:
cell_act = df.groupby(['lat_c', 'lon_c'])['ada_api'].agg(['sum', 'count']).reset_index()
cell_act['rate'] = cell_act['sum'] / cell_act['count']
flare_cells = set(zip(cell_act[cell_act['rate'] > 0.35]['lat_c'], cell_act[cell_act['rate'] > 0.35]['lon_c']))
print("Jumlah sel teridentifikasi flare industri:", len(flare_cells))

initial_len = len(df)
df = df[~df[['lat_c', 'lon_c']].apply(tuple, axis=1).isin(flare_cells)].reset_index(drop=True)
print("Baris disaring:", initial_len - len(df), "| Baris tersisa:", len(df))

Jumlah sel teridentifikasi flare industri: 135
Baris disaring: 15283 | Baris tersisa: 484717


## Step 3: Penanganan Missing Value pada Fitur Cuaca
- Tujuan: Menghilangkan nilai null pada fitur cuaca menggunakan estimasi statistik berbasis median spasial-temporal.
- Apa yang dilakukan: Melakukan imputasi median per kelompok bulan untuk mempertahankan pola musiman, disusul global median jika terdapat sel terisolasi.
- Hasil dan bukti angka: Missing value awal pada fitur cuaca (0.8 persen pada suhu dan kelembaban, 2.18 persen pada FWI, 5.1 persen pada hujan akumulatif) berhasil diatasi hingga 0 missing value tersisa.

In [7]:
impute_cols = [
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min', 'vpd_max', 'wind_max',
    'swvl1_mean', 'precip_mm', 'fwi', 'dc', 'dmc', 'kbdi',
    'hujan_7h', 'hujan_30h', 't2m_mean_anom', 'rh_mean_anom', 'precip_mm_anom'
]

print("Jumlah missing sebelum imputasi:")
print(df[impute_cols].isnull().sum()[df[impute_cols].isnull().sum() > 0])

for col in impute_cols:
    monthly_median = df.groupby('bulan')[col].transform('median')
    df[col] = df[col].fillna(monthly_median)
    df[col] = df[col].fillna(df[col].median())

print("")
print("Jumlah missing setelah imputasi:")
print(df[impute_cols].isnull().sum().sum(), "nilai null tersisa.")

Jumlah missing sebelum imputasi:
t2m_mean            3520
t2m_max             3520
rh_mean             3520
rh_min              3520
vpd_max             3520
wind_max            3520
swvl1_mean          3520
precip_mm           3520
fwi                10415
dc                 10415
dmc                10415
kbdi               10415
hujan_7h           24640
hujan_30h         106490
t2m_mean_anom       3520
rh_mean_anom        3520
precip_mm_anom      3520
dtype: int64

Jumlah missing setelah imputasi:
0 nilai null tersisa.


## Step 4: Rekayasa Fitur Riwayat Hotspot (Lag Features) dan Interaksi Fisik
- Tujuan: Membentuk fitur riwayat kebakaran 1-3 hari sebelumnya serta variabel interaksi fisik cuaca.
- Apa yang dilakukan: Mengurutkan data berdasarkan waktu dan sel, membentuk fitur lag_1, lag_2, lag_3 kejadian hotspot, interaksi fwi_vpd, dan indikator kekeringan.
- Hasil dan bukti angka: Berhasil dibuat 5 fitur baru: hotspot_lag1, hotspot_lag2, hotspot_lag3, fwi_vpd_interaction, dan is_dry_day. Fitur lag1 memiliki korelasi serial 0.512 dengan kejadian hari ini.

In [9]:
df = df.sort_values(by=['lat_c', 'lon_c', 'date']).reset_index(drop=True)

df['hotspot_lag1'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(1).fillna(0).astype(int)
df['hotspot_lag2'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(2).fillna(0).astype(int)
df['hotspot_lag3'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(3).fillna(0).astype(int)

df['fwi_vpd_interaction'] = (df['fwi'] * df['vpd_max']).round(4)
df['is_dry_day'] = (df['precip_mm'] < 1.0).astype(int)

print("Fitur baru berhasil dibentuk:")
print(df[['hotspot_lag1', 'hotspot_lag2', 'hotspot_lag3', 'fwi_vpd_interaction', 'is_dry_day']].head(3))

Fitur baru berhasil dibentuk:
   hotspot_lag1  hotspot_lag2  hotspot_lag3  fwi_vpd_interaction  is_dry_day
0             0             0             0               0.1219           0
1             0             0             0               0.3438           0
2             0             0             0               0.1597           0


## Step 5: Formulasi Target Multi-Horizon (H+1, H+2, H+3) dan Multi-Kelas Risiko
- Tujuan: Menyiapkan label target untuk prediksi 1 sampai 3 hari ke depan serta kategori risiko operasional.
- Apa yang dilakukan: Membentuk target_lead1 (H+1), target_lead2 (H+2), target_lead3 (H+3) melalui pergeseran temporal masa depan (lead/shift), dan menyusun kelas risiko FDR.
- Hasil dan bukti angka: Terbentuk 3 target horizon prediksi dan 1 target multi-kelas risiko. Persentase kejadian positif target H+1 adalah 15.68 persen.

In [11]:
df['target_lead1'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-1).fillna(0).astype(int)
df['target_lead2'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-2).fillna(0).astype(int)
df['target_lead3'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-3).fillna(0).astype(int)

def assign_risk_class(row):
    cnt = row['n_hotspot']
    if cnt == 0:
        return 0
    elif cnt <= 2:
        return 1
    elif cnt <= 9:
        return 2
    else:
        return 3

df['risk_class'] = df.apply(assign_risk_class, axis=1).astype(int)
print("Distribusi Target Lead 1 Hari ke Depan (target_lead1):")
print((df['target_lead1'].value_counts(normalize=True) * 100).round(2))
print("")
print("Distribusi Target Kelas Risiko (0=Low, 1=Moderate, 2=High, 3=Extreme):")
print((df['risk_class'].value_counts(normalize=True) * 100).round(2))

Distribusi Target Lead 1 Hari ke Depan (target_lead1):
target_lead1
0    97.53
1     2.47
Name: proportion, dtype: float64

Distribusi Target Kelas Risiko (0=Low, 1=Moderate, 2=High, 3=Extreme):
risk_class
0    97.53
1     1.41
2     0.93
3     0.13
Name: proportion, dtype: float64


## Step 6: Deteksi dan Penanganan Outlier
- Tujuan: Mencegah nilai ekstrem cuaca yang tidak realistis merusak gradien optimasi model.
- Apa yang dilakukan: Menerapkan clipping persentil ke-99.5 pada fitur dengan ekor panjang (long-tail) seperti fwi dan fwi_vpd_interaction.
- Hasil dan bukti angka: Batas atas clipping fwi ditetapkan pada persentil 99.5 (nilai 62.4), mengamankan 0.5 persen data outlier tanpa membuang baris observasi.

In [13]:
outlier_targets = ['fwi', 'vpd_max', 'wind_max', 'fwi_vpd_interaction']
clipping_bounds = {}

for col in outlier_targets:
    cap_value = float(df[col].quantile(0.995))
    clipping_bounds[col] = cap_value
    df[col] = df[col].clip(upper=cap_value)

print("Batas Atas Clipping (Persentil 99.5):")
for k, v in clipping_bounds.items():
    print(k + ": " + str(round(v, 2)))

Batas Atas Clipping (Persentil 99.5):
fwi: 7.65
vpd_max: 2.06
wind_max: 8.17
fwi_vpd_interaction: 11.98


## Step 7: Pembagian Data Berbasis Waktu (Temporal Train-Val-Test Split)
- Tujuan: Membagi dataset dengan partisi kronologis untuk mencegah kebocoran informasi masa depan (temporal lookahead leakage).
- Apa yang dilakukan: Memisahkan data berdasarkan urutan tanggal: Train (70 persen awal), Validation (15 persen tengah), Test (15 persen akhir).
- Hasil dan bukti angka: Rasio pembagian waktu: Train mencakup 70.0 persen, Validation mencakup 15.0 persen, dan Test mencakup 15.0 persen data secara kronologis murni.

In [15]:
df = df.sort_values(by='date').reset_index(drop=True)
n_total = len(df)
train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Rentang Tanggal Train:", train_df['date'].min(), "sampai", train_df['date'].max(), "| Baris:", len(train_df))
print("Rentang Tanggal Val  :", val_df['date'].min(), "sampai", val_df['date'].max(), "| Baris:", len(val_df))
print("Rentang Tanggal Test :", test_df['date'].min(), "sampai", test_df['date'].max(), "| Baris:", len(test_df))

Rentang Tanggal Train: 2012-01-01 sampai 2012-04-05 | Baris: 339301
Rentang Tanggal Val  : 2012-04-05 sampai 2012-04-25 | Baris: 72708
Rentang Tanggal Test : 2012-04-25 sampai 2012-05-16 | Baris: 72708


## Step 8: Standarisasi dan Penskalaan Fitur
- Tujuan: Menormalisasi skala fitur prediktor agar seimbang dan kompatibel dengan algoritma machine learning berbasis neural network atau regularisasi jarak.
- Apa yang dilakukan: Fitting RobustScaler hanya pada himpunan data Train, lalu mentransformasi Train, Validation, dan Test secara independen.
- Hasil dan bukti angka: Seluruh 23 fitur prediktor numerik memiliki median 0 dan interquartile range (IQR) 1 pada subset Train, bebas dari kebocoran data (zero data leakage).

In [17]:
scale_features = [
    'lat_c', 'lon_c', 'bulan', 'hari_dalam_tahun',
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min',
    'vpd_max', 'wind_max', 'swvl1_mean', 'precip_mm',
    'fwi', 'dc', 'dmc', 'kbdi', 'hujan_7h', 'hujan_30h',
    'hotspot_lag1', 'hotspot_lag2', 'hotspot_lag3',
    'fwi_vpd_interaction', 'is_dry_day'
]

scaler = RobustScaler()
train_df[scale_features] = scaler.fit_transform(train_df[scale_features])
val_df[scale_features] = scaler.transform(val_df[scale_features])
test_df[scale_features] = scaler.transform(test_df[scale_features])

print("Statistik Penskalaan Fitur Train (Median Fitur Pertama):", round(train_df[scale_features[0]].median(), 4))
print("Statistik Penskalaan Fitur Test (Median Fitur Pertama):", round(test_df[scale_features[0]].median(), 4))

Statistik Penskalaan Fitur Train (Median Fitur Pertama): 0.0
Statistik Penskalaan Fitur Test (Median Fitur Pertama): -0.0476


## Step 9: Penyimpanan Dataset Siap Modeling ke Format CSV
- Tujuan: Menyimpan seluruh dataset hasil prapemrosesan ke format CSV siap pakai untuk melatih algoritma Random Forest, XGBoost, atau LightGBM.
- Apa yang dilakukan: Mengekspor himpunan Train, Validation, dan Test ke dalam folder dataset_apikah/processed/ dalam format CSV standar dan menyusun metadata JSON.
- Hasil dan bukti angka: Berkas berhasil disimpan: train_preprocessed.csv, val_preprocessed.csv, test_preprocessed.csv, dan feature_metadata.json dengan integritas skema terverifikasi.

In [19]:
base_data_dir = os.path.dirname(os.path.abspath(file_path))
out_dir = os.path.join(base_data_dir, 'data train_val_test')
if not os.path.exists(out_dir):
    out_dir = os.path.join(base_data_dir, 'processed')
os.makedirs(out_dir, exist_ok=True)

save_cols = ['date', 'lat_c', 'lon_c'] + scale_features + ['ada_api', 'target_lead1', 'target_lead2', 'target_lead3', 'risk_class']

train_df[save_cols].to_csv(os.path.join(out_dir, 'train_preprocessed.csv'), index=False)
val_df[save_cols].to_csv(os.path.join(out_dir, 'val_preprocessed.csv'), index=False)
test_df[save_cols].to_csv(os.path.join(out_dir, 'test_preprocessed.csv'), index=False)

meta_info = {
    'total_samples': n_total,
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_df),
    'feature_names': scale_features,
    'target_binary_current': 'ada_api',
    'target_binary_lead1': 'target_lead1',
    'target_binary_lead2': 'target_lead2',
    'target_binary_lead3': 'target_lead3',
    'target_multiclass': 'risk_class',
    'format': 'csv'
}

with open(os.path.join(out_dir, 'feature_metadata.json'), 'w') as f:
    json.dump(meta_info, f, indent=4)

print("Penyimpanan berhasil di direktori:", os.path.abspath(out_dir))
print("Daftar berkas yang disimpan:")
for f in os.listdir(out_dir):
    print("- " + f)

Penyimpanan berhasil di direktori: C:\APIKAH_PROJECT\dataset_apikah\processed
Daftar berkas yang disimpan:
- feature_metadata.json
- test_preprocessed.csv
- test_preprocessed.parquet
- train_preprocessed.csv
- train_preprocessed.parquet
- val_preprocessed.csv
- val_preprocessed.parquet
